In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Pharmacovigilance/ADR.csv')

In [8]:
df

,text,drug,effect,indexes
0,Intravenous azithromycin-induced ototoxicity.,azithromycin,ototoxicity,"{'drug': {'start_char': [12], 'end_char': [24]..."
1,"Immobilization, while Paget's bone disease was...",dihydrotachysterol,increased calcium-release,"{'drug': {'start_char': [91], 'end_char': [109..."
2,Unaccountable severe hypercalcemia in a patien...,dihydrotachysterol,hypercalcemia,"{'drug': {'start_char': [84], 'end_char': [102..."
3,METHODS: We report two cases of pseudoporphyri...,naproxen,pseudoporphyria,"{'drug': {'start_char': [58], 'end_char': [66]..."
4,METHODS: We report two cases of pseudoporphyri...,oxaprozin,pseudoporphyria,"{'drug': {'start_char': [71], 'end_char': [80]..."
...,...,...,...,...
6816,Lithium treatment was terminated in 1975 becau...,Lithium,lithium intoxication,"{'drug': {'start_char': [0], 'end_char': [7]},..."
6817,Lithium treatment was terminated in 1975 becau...,lithium,lithium intoxication,"{'drug': {'start_char': [52], 'end_char': [59]..."
6818,Eosinophilia caused by clozapine was observed ...,clozapine,Eosinophilia,"{'drug': {'start_char': [23], 'end_char': [32]..."
6819,Eosinophilia has been encountered from 0.2 to ...,clozapine,Eosinophilia,"{'drug': {'start_char': [55], 'end_char': [64]..."


In [9]:
df.shape

(6821, 4)

In [10]:
print(df['text'].duplicated().sum())

2550


In [11]:
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(len(df))

4271


In [12]:
df

,text,drug,effect,indexes
0,Intravenous azithromycin-induced ototoxicity.,azithromycin,ototoxicity,"{'drug': {'start_char': [12], 'end_char': [24]..."
1,"Immobilization, while Paget's bone disease was...",dihydrotachysterol,increased calcium-release,"{'drug': {'start_char': [91], 'end_char': [109..."
2,Unaccountable severe hypercalcemia in a patien...,dihydrotachysterol,hypercalcemia,"{'drug': {'start_char': [84], 'end_char': [102..."
3,METHODS: We report two cases of pseudoporphyri...,naproxen,pseudoporphyria,"{'drug': {'start_char': [58], 'end_char': [66]..."
4,"Naproxen, the most common offender, has been a...",Naproxen,erythropoietic protoporphyria,"{'drug': {'start_char': [0], 'end_char': [8]},..."
...,...,...,...,...
4266,Reversible nonthrombocytopenic palpable purpur...,metoclopramide,nonthrombocytopenic palpable purpura,"{'drug': {'start_char': [64], 'end_char': [78]..."
4267,Lithium treatment was terminated in 1975 becau...,Lithium,diabetes insipidus-like syndrome,"{'drug': {'start_char': [0], 'end_char': [7]},..."
4268,Eosinophilia caused by clozapine was observed ...,clozapine,Eosinophilia,"{'drug': {'start_char': [23], 'end_char': [32]..."
4269,Eosinophilia has been encountered from 0.2 to ...,clozapine,Eosinophilia,"{'drug': {'start_char': [55], 'end_char': [64]..."


In [13]:
import ast

In [14]:
sentence_spans = {}

for text, indexes in zip(df["text"], df["indexes"]):

    if isinstance(indexes, str):
        indexes = ast.literal_eval(indexes)

    if text not in sentence_spans:
        sentence_spans[text] = []

    for entity_type, key in [("DRUG", "drug"), ("EFFECT", "effect")]:
        positions = indexes[key]

        for start, end in zip(positions["start_char"], positions["end_char"]):
            span = {"start": start, "end": end, "label": entity_type}

            if span not in sentence_spans[text]:
                sentence_spans[text].append(span)

ner_df = pd.DataFrame({
    "text": list(sentence_spans.keys()),
    "spans": list(sentence_spans.values())
})

print(ner_df.shape)

(4271, 2)


In [15]:
ner_df

,text,spans
0,Intravenous azithromycin-induced ototoxicity.,"[{'start': 12, 'end': 24, 'label': 'DRUG'}, {'..."
1,"Immobilization, while Paget's bone disease was...","[{'start': 91, 'end': 109, 'label': 'DRUG'}, {..."
2,Unaccountable severe hypercalcemia in a patien...,"[{'start': 84, 'end': 102, 'label': 'DRUG'}, {..."
3,METHODS: We report two cases of pseudoporphyri...,"[{'start': 58, 'end': 66, 'label': 'DRUG'}, {'..."
4,"Naproxen, the most common offender, has been a...","[{'start': 0, 'end': 8, 'label': 'DRUG'}, {'st..."
...,...,...
4266,Reversible nonthrombocytopenic palpable purpur...,"[{'start': 64, 'end': 78, 'label': 'DRUG'}, {'..."
4267,Lithium treatment was terminated in 1975 becau...,"[{'start': 0, 'end': 7, 'label': 'DRUG'}, {'st..."
4268,Eosinophilia caused by clozapine was observed ...,"[{'start': 23, 'end': 32, 'label': 'DRUG'}, {'..."
4269,Eosinophilia has been encountered from 0.2 to ...,"[{'start': 55, 'end': 64, 'label': 'DRUG'}, {'..."


In [16]:
from sklearn.model_selection import train_test_split
train, temp = train_test_split(ner_df, test_size=0.2 , random_state = 42)
val, test = train_test_split(temp, test_size = 0.5 , random_state = 42)

In [17]:
train.shape , val.shape , test.shape

((3416, 2), (427, 2), (428, 2))

In [18]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test.reset_index(drop=True))

ds = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})


In [19]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

labels = ["O", "B-DRUG", "I-DRUG", "B-EFFECT", "I-EFFECT"]

label2id = {}
for index, label in enumerate(labels):
    label2id[label] = index

id2label = {}
for index, label in enumerate(labels):
    id2label[index] = label

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)


def encode(batch):

    encoded_data = tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        return_offsets_mapping=True
    )

    offsets_list = encoded_data["offset_mapping"]
    spans_list = batch["spans"]

    all_labels = []

    for sentence_index in range(len(batch["text"])):

        token_offsets = offsets_list[sentence_index]
        sentence_spans = spans_list[sentence_index]

        token_labels = []

        for token_start, token_end in token_offsets:

            if token_start == token_end:
                token_labels.append(-100)
                continue

            current_label = "O"

            for entity in sentence_spans:

                entity_start = entity["start"]
                entity_end = entity["end"]
                entity_type = entity["label"]

                overlaps = (
                    token_start < entity_end
                    and token_end > entity_start
                )

                if overlaps:

                    if token_start <= entity_start:
                        current_label = "B-" + entity_type
                    else:
                        current_label = "I-" + entity_type

                    break

            label_id = label2id[current_label]
            token_labels.append(label_id)

        all_labels.append(token_labels)

    encoded_data["labels"] = all_labels

    encoded_data.pop("offset_mapping")

    return encoded_data


tokenized = ds.map(
    encode,
    batched=True,
    remove_columns=ds["train"].column_names
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

... (2 outputs left out)

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING

Map:   0%|          | 0/3416 [00:00<?, ? examples/s]

Map:   0%|          | 0/427 [00:00<?, ? examples/s]

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

In [23]:
sample = ds["train"][84]

print("Text:", sample["text"])
print("Spans:", sample["spans"])

Text: Case report: life-threatening hypoglycaemia associated with sulfadoxine-pyrimethamine, a commonly used antimalarial drug.
Spans: [{'end': 85, 'label': 'DRUG', 'start': 60}, {'end': 43, 'label': 'EFFECT', 'start': 13}]


In [27]:
encoded = encode({
    "text": [sample["text"]],
    "spans": [sample["spans"]]
})

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

for token, label_id in zip(tokens, encoded["labels"][0]):
    label = id2label.get(label_id, "IGNORED")
    print(token, ":", label)

[CLS] : IGNORED
case : O
report : O
: : O
life : B-EFFECT
- : I-EFFECT
threatening : I-EFFECT
hypoglycaemia : I-EFFECT
associated : O
with : O
sulf : B-DRUG
##ado : I-DRUG
##xine : I-DRUG
- : I-DRUG
pyrim : I-DRUG
##eth : I-DRUG
##amine : I-DRUG
, : O
a : O
commonly : O
used : O
antimalarial : O
drug : O
. : O
[SEP] : IGNORED
